## Fetching

In [1]:
import pandas as pd
import os
import torch
import matplotlib.pyplot as plt
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from PIL import Image
import numpy as np
import cv2
from roboflow import Roboflow
from ultralytics import YOLO
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
#s


labels_df = pd.read_csv('labels_cleaned.csv') 
labels_df['inverted'] = False
print(labels_df.head())

image_dir = '../Images\\train\\' 
def load_image(image_name):
    file_extension = '.jpeg'
    filename = f"{image_name}{file_extension}"
    file_path = os.path.join(image_dir, filename)
    return Image.open(file_path)

for i in range (1):
    sample_image = load_image(labels_df.iloc[i]['image'])
    sample_image.show()



      image  level  inverted
0   10_left      0     False
1  10_right      0     False
2   13_left      0     False
3  13_right      0     False
4   15_left      1     False


In [2]:
print(torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")  # Should print "Using device: cuda"

print("Torch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Device Count:", torch.cuda.device_count())
print("CUDA Device Name:", torch.cuda.get_device_name(0))
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

True
Using device: cuda
Torch Version: 2.5.1+cu121
CUDA Available: True
CUDA Device Count: 1
CUDA Device Name: NVIDIA GeForce RTX 3070


## YOLO version 1 training



In [ ]:

rf1 = Roboflow(api_key="YkB7AdZqhiQmoKyu0FvN")
project1 = rf1.workspace("graduation-project-f8ud5").project("my-first-project-nxzes")
version = project1.version(1)
dataset = version.download("yolov8")

# Load a model
model1 = YOLO("yolo11n.pt")  # load a pretrained model (recommended for training)

# Start training on your custom dataset
model1.train(data="G:\Graduation Project\Graduation-Project\My-First-Project-2\data.yaml", epochs=115, imgsz=640, batch=16, device=device,  workers=0)

In [ ]:
results = model1.predict(source="G:\\Graduation Project\\Images\\train", save=True, save_txt=True, conf=0.1)
print("Inference complete! Check 'runs/detect/predict/' for results.")

## YOLO version 2

In [ ]:
# Load a model
model2 = YOLO("yolo11n.pt")  # load a pretrained model (recommended for training)

# Start training on your custom dataset
model2.train(data="G:\Graduation Project\Graduation-Project\My First Project.-2\data.yaml", epochs=60, imgsz=640, batch=16, device=device,workers=0,mosaic=False)

In [ ]:
# Load your trained model (adjust the path to your weights file)
model = YOLO("G:\Graduation Project\Graduation-Project\\runs\detect\\Version -3-training-results-60epochs\\weights\\best.pt")
image_dir = "G:\\Graduation Project\\Images\\train"
save_dir = "G:\\Graduation Project\\Labled-Images-clean"


# Run inference on an image file
# results = model.predict(source="G:\\Graduation Project\\Images\\train", conf=0.3)
# results.save(save_dir="G:\\Graduation Project\\Labled-Images")
# for img_file in os.listdir(image_dir):
#     if img_file.endswith('.jpeg'):
#         img_path = os.path.join(image_dir, img_file)
#         results = model.predict(source=img_path, conf=0.3)
#         # Plot the results and save manually
#         annotated_img = results[0].plot()  # Returns annotated image as numpy array
#         output_path = os.path.join(save_dir, img_file)
#         cv2.imwrite(output_path, annotated_img)


# Verify model loading
print("Model loaded successfully. Class names:", model.names)

# Process images one by one
for img_file in os.listdir(image_dir):
    if img_file.endswith('.jpeg'):
        img_path = os.path.join(image_dir, img_file)
        print(f"Processing {img_file}")
        
        # Load and check image
        img = cv2.imread(img_path)
        if img is None:
            print(f"Error: Could not load image {img_path}")
            continue
        
        # Normalization
        img_lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(img_lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l_clahe = clahe.apply(l)
        img_lab = cv2.merge((l_clahe, a, b))
        img_normalized = cv2.cvtColor(img_lab, cv2.COLOR_LAB2BGR)
        
        # Resize
        img_resized = cv2.resize(img_normalized, (640, 640))
        
        # Denoise
        img_denoised = cv2.GaussianBlur(img_resized, (5, 5), 0)
        
        # Predict and debug detections
        results = model.predict(source=img_denoised, conf=0.25, verbose=True)
        result = results[0]
        print(f"Number of detections: {len(result.boxes)}")
        
        features = {}
        for box in result.boxes:
            coords = box.xyxy[0].tolist()
            label = result.names[int(box.cls)].lower()
            x_center = (coords[0] + coords[2]) / 2
            y_center = (coords[1] + coords[3]) / 2
            features[label] = (x_center, y_center)
            print(f"Detected {label} at ({x_center}, {y_center})")
        
        # Check if anything was detected
        if not features:
            print(f"Warning: No features detected in {img_file}")
            cv2.imwrite(os.path.join(save_dir, f"no_detection_{img_file}"), img_denoised)
            continue
        
        # Determine orientation
        is_inverted = None
        if "notch" in features:
            is_inverted = False
        elif "macula" in features and "optic_nerve" in features:
            macula_y = features["macula"][1]
            optic_nerve_y = features["optic_nerve"][1]
            is_inverted = macula_y > optic_nerve_y
        else:
            print(f"Warning: Incomplete features for orientation: {features.keys()}")

        # Apply rotation if determined
        if is_inverted is not None:
            img_final = cv2.rotate(img_denoised, cv2.ROTATE_180) if is_inverted else img_denoised
        else:
            img_final = img_denoised
        
        # Render bounding boxes on the image

        # annotated_img = result.plot()  # Generate image with boxes and labels
        annotated_img= img_final
        
        # Apply rotation to annotated image if inverted
        if is_inverted:
            annotated_img = cv2.rotate(annotated_img, cv2.ROTATE_180)
        
        # Save the annotated image
        output_path = os.path.join(save_dir, f"annotated_{img_file}")
        cv2.imwrite(output_path, annotated_img)
        print(f"Saved annotated image: {output_path}, Inverted = {is_inverted}")        

Model loaded successfully. Class names: {0: 'Optic-nerve', 1: 'macula', 2: 'notch'}
Processing 10003_left.jpeg



## Quick fix for the labels


In [ ]:
# Load original labels
labels_file = "G:\Graduation Project\Graduation-Project\labels_cleaned.csv"  # Replace with actual path
labels_df = pd.read_csv(labels_file)

# Add "annotated_" prefix to filenames
labels_df['image'] = labels_df['image'].apply(lambda x: f"annotated_{x}")

# Verify files exist in save_dir
save_dir = "G:\\Graduation Project\\Labled-Images"
missing_files = labels_df[~labels_df['image'].apply(lambda x: os.path.exists(os.path.join(save_dir, x)))]
if not missing_files.empty:
    print(f"Warning: {len(missing_files)} files not found:")
    print(missing_files['image'].tolist())
else:
    print("All annotated files found!")

# Save updated labels
labels_df.to_csv("annotated_labels.csv", index=False)
print("Updated labels saved as 'annotated_labels.csv'")

## Split data into train, val, test

In [ ]:
from sklearn.model_selection import train_test_split 
# Load updated labels
labels_df = pd.read_csv("annotated_labels.csv")

# Split into train, val, test
train_df, test_df = train_test_split(labels_df, test_size=0.1, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.222, random_state=42)  # 0.222 of 90% = 20% total

# Save splits
train_df.to_csv("train_labels.csv", index=False)
val_df.to_csv("val_labels.csv", index=False)
test_df.to_csv("test_labels.csv", index=False)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

## Tensorflow Debugging and GPU usage

In [3]:
import tensorflow as tf

# Check if GPU is being used
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

# Test if TensorFlow is using the GPU
tf.debugging.set_log_device_placement(True)

# Create a small tensor to test GPU usage
with tf.device('/GPU:0'):
    tensor = tf.constant([[1.0, 2.0], [3.0, 4.0]])
    print(tensor)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        details = tf.config.experimental.get_device_details(gpu)
        print(details)    

Num GPUs Available: 1
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:GPU:0
tf.Tensor(
[[          1           2]
 [          3           4]], shape=(2, 2), dtype=float32)
{'device_name': 'NVIDIA GeForce RTX 3070', 'compute_capability': (8, 6)}


## Simple Model Generator


In [11]:
import tensorflow as tf
from tensorflow.keras import layers,models,optimizers
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.applications import EfficientNetB0,EfficientNetB3, EfficientNetB7
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import mixed_precision
import tensorflow_addons as tfa

mixed_precision.set_global_policy('mixed_float16')


# Data augmentation for the training set 
train_datagen = ImageDataGenerator(
    rescale=1./255,            # Normalize pixel values to [0,1]
    rotation_range=20,         # Random rotation
    width_shift_range=0.1, 
    height_shift_range=0.1,
    zoom_range=0.1,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,      # Random horizontal flipping
    vertical_flip=True,         # Random vertical flipping
    fill_mode='nearest'
)

# Data augmentation for the validation set (No Augmentation, only rescaling)
val_datagen = ImageDataGenerator(
    rescale=1./255
    )

test_datagen = ImageDataGenerator(
    rescale=1./255
    )


batch_size = 16
image_size = (224, 224)
SAVE_DIR = "G:\\Graduation Project\\Labled-Images-cropped"

train_df = pd.read_csv("train_labels_fixed.csv")
train_df['level']=train_df['level'].astype(str)

validate_df = pd.read_csv("val_labels_fixed.csv")
validate_df['level']=train_df['level'].astype(str)

test_df = pd.read_csv("test_labels_fixed.csv")
test_df['level']=train_df['level'].astype(str)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=SAVE_DIR,
    x_col='image',
    y_col='level',
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical'
    
)

validate_gen = val_datagen.flow_from_dataframe(
    dataframe=validate_df,
    directory=SAVE_DIR,
    x_col='image',
    y_col='level',
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical'
)

test_gen = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=SAVE_DIR,
    x_col='image',
    y_col='level',
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical'
)

Found 24475 validated image filenames belonging to 5 classes.


Found 119 invalid image filename(s) in x_col="image". These filename(s) will be ignored.


Found 6983 validated image filenames belonging to 5 classes.
Found 3497 validated image filenames belonging to 5 classes.


Found 36 invalid image filename(s) in x_col="image". These filename(s) will be ignored.
Found 16 invalid image filename(s) in x_col="image". These filename(s) will be ignored.


In [4]:
import tensorflow as tf
tf.keras.backend.clear_session()

## EfficientNetB0, 224*224


In [ ]:
base_model =EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    input_shape=(224,224,3),
    pooling=None,
    classes=5,
    classifier_activation="softmax",
)

base_model.trainable = True

model = tf.keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(5, activation='softmax')
])

adam_opt=Adam(learning_rate=0.0001) #optimizer, a bit similar to gradient descent

model.compile(optimizer=adam_opt,loss='categorical_crossentropy',metrics=['accuracy'])

best_model_fie ='G:/Graduation Project/Graduation-Project/EfficientNetv2.h5'

callbacks = [
    ModelCheckpoint(
        best_model_fie, 
        monitor='val_accuracy', 
        save_best_only=True, 
        save_weights_only=True, 
        mode='max', 
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', 
        patience=25, 
        mode='max', 
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_accuracy', 
        factor=0.5, 
        patience=2, 
        verbose=1
    )
]

result = model.fit(
    train_gen,
    epochs=15,
    validation_data=validate_gen,
    callbacks=callbacks
)

best_val_acc_epoch = np.argmax(result.history['val_accuracy'])
best_val_acc = result.history['val_accuracy'][best_val_acc_epoch]

print("best validation accuarcy is : " + str(best_val_acc))

plt.plot(result.history['accuracy'],label='train acc')
plt.plot(result.history['val_accuracy'],label='val acc')
plt.legend()
plt.show()


plt.plot(result.history['loss'],label='train loss')
plt.plot(result.history['val_loss'],label='val loss')
plt.legend()
plt.show()


## EfficientNetB7, 320*320


In [ ]:
base_model =EfficientNetB7(
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    input_shape=(240,240,3),
    pooling=None,
    classes=5,
    classifier_activation="softmax",
)

base_model.trainable = True

model = tf.keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(5, activation='softmax')
])

adam_opt=Adam(learning_rate=1e-5) #optimizer, a bit similar to gradient descent

model.compile(optimizer=adam_opt,loss='categorical_crossentropy',metrics=['accuracy'])

best_model_fie ='G:/Graduation Project/Graduation-Project/EfficientNetB7.h5'

callbacks = [
    ModelCheckpoint(
        best_model_fie, 
        monitor='val_accuracy', 
        save_best_only=True, 
        save_weights_only=True, 
        mode='max', 
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', 
        patience=25, 
        mode='max', 
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_accuracy', 
        factor=0.5, 
        patience=2, 
        verbose=1
    )
]

result = model.fit(
    train_gen,
    epochs=15,
    validation_data=validate_gen,
    callbacks=callbacks
)

best_val_acc_epoch = np.argmax(result.history['val_accuracy'])
best_val_acc = result.history['val_accuracy'][best_val_acc_epoch]

print("best validation accuarcy is : " + str(best_val_acc))

plt.plot(result.history['accuracy'],label='train acc')
plt.plot(result.history['val_accuracy'],label='val acc')
plt.legend()
plt.show()


plt.plot(result.history['loss'],label='train loss')
plt.plot(result.history['val_loss'],label='val loss')
plt.legend()
plt.show()


## EfficientNetB3, 380*380


In [ ]:
base_model =EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    input_shape=(380,380,3),
    pooling=None,
    classes=5,
    classifier_activation="softmax",
)

base_model.trainable = True

model = tf.keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(5, activation='softmax',dtype='float32')
])

adam_opt=Adam(learning_rate=0.0001) #optimizer, a bit similar to gradient descent

model.compile(optimizer=adam_opt,loss='categorical_crossentropy',metrics=['accuracy'])

best_model_fie ='G:/Graduation Project/Graduation-Project/EfficientNetv2.h5'

callbacks = [
    ModelCheckpoint(
        best_model_fie, 
        monitor='val_accuracy', 
        save_best_only=True, 
        save_weights_only=True, 
        mode='max', 
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', 
        patience=25, 
        mode='max', 
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_accuracy', 
        factor=0.5, 
        patience=2, 
        verbose=1
    )
]

result = model.fit(
    train_gen,
    epochs=15,
    validation_data=validate_gen,
    callbacks=callbacks
)

best_val_acc_epoch = np.argmax(result.history['val_accuracy'])
best_val_acc = result.history['val_accuracy'][best_val_acc_epoch]

print("best validation accuarcy is : " + str(best_val_acc))

plt.plot(result.history['accuracy'],label='train acc')
plt.plot(result.history['val_accuracy'],label='val acc')
plt.legend()
plt.show()


plt.plot(result.history['loss'],label='train loss')
plt.plot(result.history['val_loss'],label='val loss')
plt.legend()
plt.show()


## EfficientNetB3, 224*224, LAYER FREEZING

In [ ]:
from tensorflow.keras import mixed_precision
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

mixed_precision.set_global_policy('mixed_float16')

base_model = EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3),
)
base_model.trainable = False  # Freeze initially

model = tf.keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(5, activation='softmax', dtype='float32')  # dtype float32 for stable loss calculation
])

# Step 1: Compile with higher LR
adam_opt=Adam(learning_rate=1e-3) #optimizer, a bit similar to gradient descent
model.compile(optimizer=adam_opt, loss='categorical_crossentropy', metrics=['accuracy'])

initial_epochs = 10
model.fit(train_gen, epochs=initial_epochs, validation_data=validate_gen)

# Step 2: Unfreeze base_model and fine-tune
base_model.trainable = True

adam_opt2=Adam(learning_rate=1e-5)
model.compile(optimizer=adam_opt2, loss='categorical_crossentropy', metrics=['accuracy'])



best_model_file ='G:/Graduation Project/Graduation-Project/EfficientNetB3224.h5'

callbacks = [
    ModelCheckpoint(
                    best_model_file, 
                    monitor='val_accuracy', 
                    save_best_only=True, 
                    mode='max',
                    verbose=1
                    ),

    EarlyStopping(
                  monitor='val_accuracy', 
                  patience=10, 
                  mode='max',
                  verbose=1
                  ),

    ReduceLROnPlateau(
                      monitor='val_accuracy', 
                      factor=0.2, 
                      patience=3, 
                      verbose=1, 
                      min_lr=1e-7
                      )
]

fine_tune_epochs = 25

class_weights = {
    0: 1.0,
    1: 10.0,
    2: 10.0,
    3: 20.0,
    4: 20.0
}


result = model.fit(
    train_gen,
    epochs=fine_tune_epochs,
    validation_data=validate_gen,
    class_weight=class_weights,
    callbacks=callbacks
)


# Evaluate best performance

best_val_acc_epoch = np.argmax(result.history['val_accuracy'])
best_val_acc = result.history['val_accuracy'][best_val_acc_epoch]

print(f"Best Validation Accuracy: {best_val_acc:.4f} at epoch {best_val_acc_epoch+1}")




# Plot accuracy
plt.figure(figsize=(8,4))
plt.plot(result.history['accuracy'], label='Train Accuracy')
plt.plot(result.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy over epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()
plt.show()

# Plot loss
plt.figure(figsize=(8,4))
plt.plot(result.history['loss'], label='Train Loss')
plt.plot(result.history['val_loss'], label='Validation Loss')
plt.title('Loss over epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()


Epoch 1/10
 131/1530 [=>............................] - ETA: 6:20 - loss: 0.8962 - accuracy: 0.7495

KeyboardInterrupt: 

## EfficientNet Enhanced Approach

In [15]:
# Build the EfficientNetB3 base
base_model = EfficientNetB3(
    include_top=False,      # We'll add our own final layers
    weights='imagenet',     # Use ImageNet-pretrained weights
    input_shape=(224,224,3)
)

# Freeze the base model initially
base_model.trainable = False

# Create your top layers
x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.BatchNormalization()(x)        # Helps stabilize training
x = layers.Dropout(0.4)(x)               # Regularization
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)

# Important if using mixed precision: ensure final layer is float32
# If not using mixed precision, you can omit "dtype='float32'"
outputs = layers.Dense(5, activation='softmax', dtype='float32')(x)

# Build the final model
model = models.Model(inputs=base_model.input, outputs=outputs)

model.summary()  # Check the architecture

cclass_weights = {
    0: 3.0,
    1: 10.0,
    2: 10.0,
    3: 20.0,
    4: 20.0
}

# Option 1: Standard cross-entropy + class weights
loss_fn = 'categorical_crossentropy'

# Option 2 (uncomment to use focal loss)
# loss_fn = tfa.losses.SigmoidFocalCrossEntropy(gamma=2.0)

optimizer = optimizers.Adam(learning_rate=1e-3)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['accuracy', tfa.metrics.CohenKappa(num_classes=5, sparse_labels=False)]
)

warmup_epochs = 5
model.fit(
    train_gen,
    epochs=warmup_epochs,
    validation_data=validate_gen,
    class_weight=class_weights,  
    verbose=1
)

# Unfreeze some layers in base_model
# Example: unfreeze the last 60 layers
for layer in base_model.layers[-60:]:
    layer.trainable = True

# Recompile with a smaller LR
fine_tune_lr = 1e-5
optimizer_finetune = optimizers.Adam(learning_rate=fine_tune_lr)

model.compile(
    optimizer=optimizer_finetune,
    loss=loss_fn,
    metrics=['accuracy', tfa.metrics.CohenKappa(num_classes=5, sparse_labels=False)]
)

# Callbacks to save best model, stop early, and reduce LR on plateau
callbacks = [
    ModelCheckpoint("best_model.h5", monitor="val_accuracy",
                    save_best_only=True, mode="max", verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=8, 
                  mode="max", restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_accuracy", factor=0.2, 
                      patience=3, min_lr=1e-7, verbose=1)
]

fine_tune_epochs = 25

# If using cross-entropy + class weights:
class_weights = {0:1., 1:5., 2:5., 3:10., 4:10.}

history = model.fit(
    train_gen,
    epochs=fine_tune_epochs,
    validation_data=validate_gen,
    class_weight=class_weights, 
    callbacks=callbacks,
    verbose=1
)


Model: "model_5"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_7 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 rescaling_12 (Rescaling)       (None, 224, 224, 3)  0           ['input_7[0][0]']                
                                                                                                  
 normalization_6 (Normalization  (None, 224, 224, 3)  7          ['rescaling_12[0][0]']           
 )                                                                                                
                                                                                            

KeyboardInterrupt: 

## Comfisuon Matrix


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Predict probabilities for validation set
y_pred_probs = model.predict(validate_gen)

# Convert predictions probabilities to class labels
y_pred_classes = np.argmax(y_pred_probs, axis=1)

# True classes
y_true = validate_gen.classes

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

# Class labels (modify if your labels differ)
class_labels = ['0', '1', '2', '3', '4']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)

plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix for Diabetic Retinopathy Detection')
plt.show()

print("Classification Report:\n", classification_report(y_true, y_pred_classes, target_names=class_labels))

In [ ]:
# Load your training labels
train_df = pd.read_csv("train_labels_fixed.csv", dtype={'level': str})

# Count samples per class
class_counts = train_df['level'].value_counts()
print(class_counts)

NameError: name 'pd' is not defined

In [ ]:
train_df = pd.read_csv("train_labels.csv")
train_df['level'] = train_df['level'].astype(str)
train_df.to_csv("train_labels_fixed.csv", index=False)
train_df['image'] = train_df['image'].apply(lambda x: x + '.jpeg' if not x.endswith('.jpeg') else x)
train_df.to_csv("train_labels_fixed.csv", index=False)

fixed_train = pd.read_csv("train_labels_fixed.csv", dtype={'level': str})
print("Train fixed dtypes:")
print(fixed_train.dtypes)
print(fixed_train.head())

val_df = pd.read_csv("val_labels.csv")
val_df['level'] = val_df['level'].astype(str)
val_df.to_csv("val_labels_fixed.csv", index=False)
val_df['image'] = val_df['image'].apply(lambda x: x + '.jpeg' if not x.endswith('.jpeg') else x)
val_df.to_csv("val_labels_fixed.csv", index=False)

fixed_val = pd.read_csv("val_labels_fixed.csv", dtype={'level': str})
print("Val fixed dtypes:")
print(fixed_val.dtypes)
print(fixed_val.head())

test_df = pd.read_csv("test_labels.csv")
test_df['level'] = train_df['level'].astype(str)
test_df['image'] = train_df['image'].apply(lambda x: x + '.jpeg' if not x.endswith('.jpeg') else x)
test_df.to_csv("test_labels_fixed.csv", index=False)

fixed_test = pd.read_csv("train_labels_fixed.csv", dtype={'level': str})
print("Test fixed dtypes:")
print(fixed_test.dtypes)
print(fixed_test.head())


In [11]:
print("Classification Report:\n", classification_report(y_true, y_pred_classes, target_names=class_labels))


Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.90      0.81      5066
           1       0.00      0.00      0.00       498
           2       0.16      0.08      0.11      1087
           3       0.01      0.01      0.01       176
           4       0.03      0.02      0.02       156

    accuracy                           0.67      6983
   macro avg       0.19      0.20      0.19      6983
weighted avg       0.55      0.67      0.60      6983



Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


# Image Cropping, and getting rid of the black borders


In [10]:
import cv2
import numpy as np
import os

def crop_and_center_retina(image_path, output_path, desired_size=300):
    """
    1) Loads an image from `image_path`.
    2) Detects and removes black borders around the retina.
    3) Resizes (or center-crops) to a fixed square (desired_size x desired_size).
    4) Saves the processed image to `output_path`.
    """
    # Load image (BGR in OpenCV)
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not load {image_path}")
        return

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Threshold to find non-black regions (tweak threshold if needed)
    _, thresh = cv2.threshold(gray, 5, 255, cv2.THRESH_BINARY)

    # Find the bounding rect of the non-black region
    coords = cv2.findNonZero(thresh)  # returns x,y coordinates of all non-black pixels
    x, y, w, h = cv2.boundingRect(coords)

    # Crop out the black border
    cropped = img[y:y+h, x:x+w]

    # Option A: Directly resize to the desired square
    # (simple, ensures consistent size, but may distort aspect ratio slightly)
    resized = cv2.resize(cropped, (desired_size, desired_size), interpolation=cv2.INTER_AREA)

    # Save processed image
    cv2.imwrite(output_path, resized)


# Example usage for a directory of images:
input_dir = "G:\Graduation Project\Labled-images-clean"
output_dir = "G:\Graduation Project\labled-images-crooped"
os.makedirs(output_dir, exist_ok=True)

for filename in os.listdir(input_dir):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)
        
        crop_and_center_retina(input_path, output_path, desired_size=300)
